# nb_mdf_06_onboard_new_source - "insert rows, not pipelines"

The key scaling message for a metadata-driven framework: adding source #101 should be a **configuration change**,
not a new pipeline. Every pack lands an extra daily feed (pipe-delimited text, dd/MM/yyyy dates) from day 1 that the
framework ignores because it has no metadata. This notebook onboards it:

1. inserts the metadata rows (`meta.ingestion_object`, `meta.column_map`, `meta.dq_rule`)
2. runs the *existing* generic Bronze and Silver notebooks for the new object - no new code
3. adds a Gold transform row and builds it

In production steps 1 and 3 are a pull request against `config/<pack>.json` in Azure DevOps, deployed by CI/CD.


In [ ]:
# no parameters - the feed to onboard comes from meta.setting (loaded by nb_mdf_00)


In [ ]:
%run nb_mdf_utils

In [ ]:
NEW = json.loads(get_setting("onboard_spec"))
oid = NEW["object"]["object_id"]
print("Onboarding", oid, "-", NEW["object"]["description"])
print("Landing folder:", NEW["object"]["landing_path"])


### Step 1 - insert the metadata rows (idempotent: delete + insert by key)

In [ ]:
def upsert_rows(table, key_col, key_val, rows_df):
    DeltaTable.forName(spark, tname("meta", table)).delete(F.col(key_col) == key_val)
    rows_df.write.mode("append").saveAsTable(tname("meta", table))

io_schema = spark.table(tname("meta", "ingestion_object")).schema
upsert_rows("ingestion_object", "object_id", oid,
            spark.createDataFrame([tuple(NEW["object"].get(f.name) for f in io_schema.fields)], io_schema))

cm_schema = spark.table(tname("meta", "column_map")).schema
cm_rows = [dict(object_id=oid, column_order=i, source_column=s, target_column=t, data_type=d, transform_expr=e,
                pii_treatment="none", sensitivity=NEW["object"]["sensitivity"], description="")
           for i, (s, t, d, e) in enumerate(NEW["columns"])]
upsert_rows("column_map", "object_id", oid,
            spark.createDataFrame([tuple(r.get(f.name) for f in cm_schema.fields) for r in cm_rows], cm_schema))

dq_schema = spark.table(tname("meta", "dq_rule")).schema
dq_rows = [dict(rule_id=r[0], object_id=oid, rule_name=r[1], rule_category=r[2], check_type=r[3], column_name=r[4],
                params=r[5], severity=r[6], fail_threshold_pct=None, rule_owner=NEW["object"]["data_owner"],
                description="", active=True) for r in NEW["rules"]]
upsert_rows("dq_rule", "object_id", oid,
            spark.createDataFrame([tuple(r.get(f.name) for f in dq_schema.fields) for r in dq_rows], dq_schema))
print("metadata inserted")

### Step 2 - run the *existing* generic notebooks for the new object - no new code, no new pipeline

In [ ]:
print(notebookutils.notebook.run("nb_mdf_10_landing_to_bronze", 1800, {"object_ids": oid}))
print(notebookutils.notebook.run("nb_mdf_20_bronze_to_silver", 1800, {"object_ids": oid}))
display(spark.table(tname("silver", NEW["object"]["silver_table"])).limit(20))

### Step 3 - add a Gold transform row and build it

In [ ]:
g = NEW["gold"]
g["sql_text"] = " ".join(g["sql_text"].split())
gt_schema = spark.table(tname("meta", "gold_transform")).schema
upsert_rows("gold_transform", "transform_id", g["transform_id"],
            spark.createDataFrame([tuple(g.get(f.name) for f in gt_schema.fields)], gt_schema))
print(notebookutils.notebook.run("nb_mdf_30_silver_to_gold", 1800, {"transform_ids": g["transform_id"]}))
display(spark.table(tname("gold", g["gold_table"])).limit(50))